# Importing libraries

In [3]:
# Basic libraries
import pandas as pd
import numpy as np
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.multioutput import MultiOutputClassifier

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss
from memory_profiler import memory_usage


# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [4]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [5]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

train = ds['train'].to_pandas()
test = ds['test'].to_pandas() 
val = ds['validation'].to_pandas()

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163168 entries, 0 to 163167
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   id             163168 non-null  object             
 1   title          163168 non-null  object             
 2   abstract       163168 non-null  object             
 3   categories     163168 non-null  object             
 4   creation_date  163168 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), object(4)
memory usage: 6.2+ MB


# Dataset preprocessing

In [6]:
allowed_categories = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

train['categories'] = train['categories'].apply(clean_element)
test['categories'] = test['categories'].apply(clean_element)
val['categories'] = val['categories'].apply(clean_element)

In [7]:
train = train[train['categories'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test = test[test['categories'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val = val[val['categories'].apply(lambda cats: all(c in allowed_categories for c in cats))]

In [8]:
train.rename(columns={'categories': 'labels'}, inplace=True)
test.rename(columns={'categories': 'labels'}, inplace=True)
val.rename(columns={'categories': 'labels'}, inplace=True)

train.rename(columns={'title': 'text'}, inplace=True)
test.rename(columns={'title': 'text'}, inplace=True)
val.rename(columns={'title': 'text'}, inplace=True)

train.drop(columns=['id', 'abstract', 'creation_date'], inplace=True)
test.drop(columns=['id', 'abstract', 'creation_date'], inplace=True)
val.drop(columns=['id', 'abstract', 'creation_date'], inplace=True)

train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)
val.reset_index(drop=True, inplace=True)

train

,text,labels
0,Upper and Lower Bounds for Large Scale Multist...,[math.OC]
1,Binary Classification: Counterbalancing Class ...,[cs.LG]
2,Smooth Optimization with Approximate Gradient,[math.OC]
3,An AI-powered Smart Routing Solution for Payme...,[cs.AI]
4,A linearly convergent method for solving high-...,[math.OC]
...,...,...
10141,Simple Question Answering with Subgraph Rankin...,"[cs.CL, cs.LG, stat.ML]"
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...","[cs.AI, cs.LG, stat.ML]"
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,"[cs.AI, cs.CL]"
10144,Near-optimal bounds for phase synchronization,[math.OC]


In [9]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train['labels'])
val_labels_binarized = mlb.transform(val['labels'])
test_labels_binarized = mlb.transform(test['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train = pd.concat([train, train_labels_df], axis=1)
val = pd.concat([val, val_labels_df], axis=1)
test = pd.concat([test, test_labels_df], axis=1)

train = train.drop(columns=['labels'])
val = val.drop(columns=['labels'])
test = test.drop(columns=['labels'])

train

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML
0,Upper and Lower Bounds for Large Scale Multist...,0,0,0,1,0
1,Binary Classification: Counterbalancing Class ...,0,0,1,0,0
2,Smooth Optimization with Approximate Gradient,0,0,0,1,0
3,An AI-powered Smart Routing Solution for Payme...,1,0,0,0,0
4,A linearly convergent method for solving high-...,0,0,0,1,0
...,...,...,...,...,...,...
10141,Simple Question Answering with Subgraph Rankin...,0,1,1,0,1
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...",1,0,1,0,1
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,1,1,0,0,0
10144,Near-optimal bounds for phase synchronization,0,0,0,1,0


In [10]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [11]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': MultiOutputClassifier(SVC()),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultiOutputClassifier(MultinomialNB()),
        'params': {
            'estimator__alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': MultiOutputClassifier(LogisticRegression(max_iter=1000)),
        'params': {
            'estimator__C': [0.1, 1, 10],
            'estimator__penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': MultiOutputClassifier(GradientBoostingClassifier()),
        'params': {
            'estimator__n_estimators': [100, 150, 200],
            'estimator__criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': MultiOutputClassifier(AdaBoostClassifier()),
        'params': {
            'estimator__n_estimators': [50, 100, 150],
            'estimator__learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': MultiOutputClassifier(SGDClassifier()),
        'params': {
            'estimator__alpha': [0.0001, 0.001, 0.01],
            'estimator__penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [12]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'hamming_loss', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = list(train.columns[1:])
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [13]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train[classes])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                val_classes = val.drop(columns=['text'])

                accuracy = accuracy_score(val_classes, y_pred)
                
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val_classes.to_numpy(), y_pred, average=None, zero_division=0)

                hamm_loss = hamming_loss(val_classes.to_numpy(), y_pred)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'hamming_loss': hamm_loss,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_multilabel3.csv', index=False)

Processing seed: 2
Processing vectorizer: TfidfVectorizer
Processing model: RandomForest
Training RandomForest with params {'n_estimators': 50, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 8.043998300003295
Peak memory usage during training: 760.36328125 MB
Prediction time: 1.023780200004694
Peak memory usage during prediction: 752.94140625 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'entropy'} and vectorizer TfidfVectorizer


C:\Users\Rafael\AppData\Local\Temp\ipykernel_21216\2813873047.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, result], ignore_index=True)


Training time: 8.664699100001599
Peak memory usage during training: 759.37890625 MB
Prediction time: 0.9895920999988448
Peak memory usage during prediction: 752.8125 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 50, 'criterion': 'log_loss'} and vectorizer TfidfVectorizer
Training time: 8.462641300000541
Peak memory usage during training: 759.15234375 MB
Prediction time: 0.9926725999976043
Peak memory usage during prediction: 759.33984375 MB
----------------------------------------------------------------------------------------------------
Training RandomForest with params {'n_estimators': 100, 'criterion': 'gini'} and vectorizer TfidfVectorizer
Training time: 15.037505100000999
Peak memory usage during training: 839.24609375 MB
Prediction time: 1.0487372999996296
Peak memory usage during prediction: 831.6328125 MB
----------------------------------------------------------------

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\neighbors\_base.py:583: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


Training time: 1.0406673000034061
Peak memory usage during training: 914.46484375 MB
Prediction time: 1.2382078000009642
Peak memory usage during prediction: 1110.98046875 MB
----------------------------------------------------------------------------------------------------
Training KNeighbors with params {'n_neighbors': 3, 'algorithm': 'kd_tree'} and vectorizer TfidfVectorizer


...

Training time: 5.848965099998168
Peak memory usage during training: 940.85546875 MB
Prediction time: 1.186997100005101
Peak memory usage during prediction: 939.50390625 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'estimator__n_estimators': 150, 'estimator__learning_rate': 0.1} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to ci

Training time: 5.848207600000023
Peak memory usage during training: 941.015625 MB
Prediction time: 1.2018731999996817
Peak memory usage during prediction: 939.83984375 MB
----------------------------------------------------------------------------------------------------
Training AdaBoost with params {'estimator__n_estimators': 150, 'estimator__learning_rate': 1.0} and vectorizer CountVectorizer


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to ci

Training time: 5.842980599998555
Peak memory usage during training: 941.4140625 MB
Prediction time: 1.2174203000031412
Peak memory usage during prediction: 939.85546875 MB
----------------------------------------------------------------------------------------------------
Processing model: SGD
Training SGD with params {'estimator__alpha': 0.0001, 'estimator__penalty': 'l2'} and vectorizer CountVectorizer
Training time: 1.2232220000005327
Peak memory usage during training: 941.10546875 MB
Prediction time: 1.8156564999953844
Peak memory usage during prediction: 936.796875 MB
----------------------------------------------------------------------------------------------------
Training SGD with params {'estimator__alpha': 0.0001, 'estimator__penalty': 'l1'} and vectorizer CountVectorizer
Training time: 0.6964176000037696
Peak memory usage during training: 941.42578125 MB
Prediction time: 1.7963613000029
Peak memory usage during prediction: 936.6875 MB
---------------------------------------

# Process results

In [14]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   seed                     396 non-null    object 
 1   vectorizer               396 non-null    object 
 2   model                    396 non-null    object 
 3   params                   396 non-null    object 
 4   accuracy                 396 non-null    float64
 5   hamming_loss             396 non-null    float64
 6   training_time            396 non-null    float64
 7   prediction_time          396 non-null    float64
 8   peak_memory_train        396 non-null    float64
 9   peak_memory_prediction   396 non-null    float64
 10  precision_class_cs.AI    396 non-null    float64
 11  recall_class_cs.AI       396 non-null    float64
 12  f1_class_cs.AI           396 non-null    float64
 13  precision_class_cs.CL    396 non-null    float64
 14  recall_class_cs.CL       3

In [15]:
results.head()

,seed,vectorizer,model,params,accuracy,hamming_loss,training_time,prediction_time,peak_memory_train,peak_memory_prediction,...,f1_class_cs.CL,precision_class_cs.LG,recall_class_cs.LG,f1_class_cs.LG,precision_class_math.OC,recall_class_math.OC,f1_class_math.OC,precision_class_stat.ML,recall_class_stat.ML,f1_class_stat.ML
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.397314,0.175829,8.043998,1.023780,760.363281,752.941406,...,0.827109,0.748596,0.816233,0.780952,0.829787,0.544186,0.657303,0.690323,0.335423,0.451477
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.402054,0.173302,8.664699,0.989592,759.378906,752.812500,...,0.826739,0.773212,0.777948,0.775573,0.835714,0.544186,0.659155,0.731544,0.341693,0.465812
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.402054,0.173302,8.462641,0.992673,759.152344,759.339844,...,0.826739,0.773212,0.777948,0.775573,0.835714,0.544186,0.659155,0.731544,0.341693,0.465812
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.408373,0.172670,15.037505,1.048737,839.246094,831.632812,...,0.832418,0.759207,0.820827,0.788815,0.848921,0.548837,0.666667,0.683544,0.338558,0.452830
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.403633,0.171090,16.582600,1.044874,838.863281,831.261719,...,0.829001,0.775602,0.788668,0.782080,0.865672,0.539535,0.664756,0.728571,0.319749,0.444444


In [16]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,hamming_loss,training_time,prediction_time,peak_memory_train,peak_memory_prediction,...,precision_class_cs.LG,recall_class_cs.LG,f1_class_cs.LG,precision_class_math.OC,recall_class_math.OC,f1_class_math.OC,precision_class_stat.ML,recall_class_stat.ML,f1_class_stat.ML,f1_avg
0,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.103476,0.263665,4.056924,1.096922,932.381510,930.588542,...,0.762069,0.338438,0.468717,1.000000,0.018605,0.036530,0.000000,0.000000,0.000000,0.183540
1,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.230648,0.221327,4.041683,1.094428,932.286458,930.667969,...,0.776596,0.558959,0.650045,0.835821,0.260465,0.397163,0.863636,0.059561,0.111437,0.383474
2,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 100, 'estimator__l...",3.333333,0.346761,0.184518,4.033176,1.089892,932.537760,930.640625,...,0.787931,0.699847,0.741281,0.798507,0.497674,0.613181,0.604167,0.272727,0.375810,0.590431
3,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.121643,0.257504,5.819623,1.183669,932.369792,930.842448,...,0.772871,0.375191,0.505155,1.000000,0.018605,0.036530,0.000000,0.000000,0.000000,0.202367
4,AdaBoost,CountVectorizer,"{'estimator__n_estimators': 150, 'estimator__l...",3.333333,0.259084,0.209795,5.831298,1.187966,932.725260,931.061198,...,0.784047,0.617152,0.690660,0.844156,0.302326,0.445205,0.826087,0.059561,0.111111,0.435172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'rbf'}",3.333333,0.406003,0.161453,21.928254,2.398623,1170.006510,940.446615,...,0.810937,0.794793,0.802784,0.897059,0.567442,0.695157,0.725191,0.297806,0.422222,0.625141
128,SVC,TfidfVectorizer,"{'estimator__C': 1, 'estimator__kernel': 'sigm...",3.333333,0.408373,0.165087,12.973877,1.492002,1139.014323,938.011719,...,0.778614,0.791730,0.785118,0.836364,0.641860,0.726316,0.647368,0.385580,0.483301,0.650478
129,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'lin...",3.333333,0.346761,0.203160,17.302771,1.391400,1177.936198,937.553385,...,0.719219,0.733538,0.726308,0.753927,0.669767,0.709360,0.497006,0.520376,0.508423,0.648333
130,SVC,TfidfVectorizer,"{'estimator__C': 10, 'estimator__kernel': 'rbf'}",3.333333,0.413112,0.163981,47.800119,2.546013,1168.036458,941.411458,...,0.788110,0.791730,0.789916,0.880795,0.618605,0.726776,0.621359,0.401254,0.487619,0.662213


In [17]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train[classes])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test[classes], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test[classes], y_pred, average=None, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: MultinomialNB
Best model params: {'estimator__alpha': 0.1}
Best vectorizer: CountVectorizer
Best accuracy: 0.39448818897637794

Class cs.AI
Precision: 0.4788273615635179
Recall: 0.4336283185840708
F1: 0.4551083591331269
Support: 339

Class cs.CL
Precision: 0.87409200968523
Recall: 0.891358024691358
F1: 0.882640586797066
Support: 405

Class cs.LG
Precision: 0.7627365356622998
Recall: 0.7987804878048781
F1: 0.7803425167535368
Support: 656

Class math.OC
Precision: 0.748898678414097
Recall: 0.7870370370370371
F1: 0.7674943566591422
Support: 216

Class stat.ML
Precision: 0.5444126074498568
Recall: 0.5993690851735016
F1: 0.5705705705705706
Support: 317



In [18]:
with open('models/best_model_sklearn_multilabel3.pkl', 'wb') as f:
    pickle.dump(pipeline, f)